# Data Loading and Feature Engineering

This notebook loads the latest local OpenMementos feature build by default and optionally regenerates trace-level and block-level parquet partitions.

Full regeneration streams the Hugging Face dataset, parses each row into engineered trace-level and block-level feature tables, and saves local parquet files under `data/`. The `data/` directory is ignored by Git, so these generated files are not committed.

Keep `RUN_FULL_BUILD = False` unless deliberately rebuilding the local parquet files. The output is partitioned into multiple parquet files to avoid holding the full dataset in memory.

In [ ]:
import sys
from collections.abc import Mapping
from datetime import datetime
from itertools import islice
from pathlib import Path

from datasets import load_dataset
import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src.reasoning_compression.features import (
    DATASET_ID,
    SPLIT,
    latest_feature_build_dir,
    normalize_difficulty,
    write_feature_partitions,
)

# Number of original traces processed before writing one parquet partition.
CHUNK_SIZE = 10_000

# Full builds stream OpenMementos and can take hours. Keep this False unless
# deliberately regenerating local parquet files.
RUN_FULL_BUILD = False

# Use a small value for testing, then set to None for the full dataset.
# MAX_ROWS = 1_000
MAX_ROWS = None

In [ ]:
FEATURE_BUILDS_DIR = Path("../data/full_feature_builds")

if RUN_FULL_BUILD:
    # Create a run-specific output directory so full builds do not overwrite
    # earlier runs.
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUTPUT_DIR = FEATURE_BUILDS_DIR / RUN_ID
    TRACE_DIR = OUTPUT_DIR / "traces"
    BLOCK_DIR = OUTPUT_DIR / "blocks"

    TRACE_DIR.mkdir(parents=True, exist_ok=True)
    BLOCK_DIR.mkdir(parents=True, exist_ok=True)
else:
    OUTPUT_DIR = latest_feature_build_dir(FEATURE_BUILDS_DIR)
    TRACE_DIR = OUTPUT_DIR / "traces"
    BLOCK_DIR = OUTPUT_DIR / "blocks"

OUTPUT_DIR

In [ ]:
# Safety check: use the existing local feature build by default.
RUN_FULL_BUILD

In [ ]:
# Stream the dataset and write trace/block parquet partitions only when
# explicitly requested. The default path uses the latest complete local build.
if RUN_FULL_BUILD:
    build_summary = write_feature_partitions(
        output_dir=OUTPUT_DIR,
        chunk_size=CHUNK_SIZE,
        max_rows=MAX_ROWS,
    )
else:
    build_summary = {
        "output_dir": str(OUTPUT_DIR),
        "status": "using_existing_local_build",
    }

build_summary

In [18]:
df_traces_full_preview = pd.read_parquet(TRACE_DIR)
df_blocks_full_preview = pd.read_parquet(BLOCK_DIR)

df_traces_full_preview.shape, df_blocks_full_preview.shape

((228557, 19), (2013510, 21))

In [10]:
# Downstream cells read the selected local build from OUTPUT_DIR.
# Keep RUN_FULL_BUILD = False unless deliberately regenerating partitions.

In [ ]:
# Load block-level features from the selected local build.
df_blocks_full = pd.read_parquet(BLOCK_DIR)
df_blocks_full["difficulty"] = df_blocks_full["difficulty"].map(
    normalize_difficulty
)

# Define the high-compression target globally on the full local feature build.
compression_threshold = df_blocks_full["summary_to_block_token_ratio"].quantile(0.25)

df_blocks_full["high_token_compression"] = (
    df_blocks_full["summary_to_block_token_ratio"] <= compression_threshold
).astype(int)

compression_threshold, df_blocks_full["high_token_compression"].value_counts(normalize=True).round(4)

In [20]:
# Save a labeled block-level table for modeling notebooks.
df_blocks_full.to_parquet(OUTPUT_DIR / "blocks_features_full_labeled.parquet", index=False)

In [ ]:
# Load trace-level features from the selected local build.
df_traces_full = pd.read_parquet(TRACE_DIR)
df_traces_full["difficulty"] = df_traces_full["difficulty"].map(
    normalize_difficulty
)

# Aggregate block-level compression behavior to the trace level. This creates
# one compression summary row per original reasoning trace.
trace_compression = (
    df_blocks_full
    .groupby("trace_id")
    .agg(
        trace_block_tokens=("block_tokens", "sum"),
        trace_summary_tokens=("summary_tokens", "sum"),
        trace_mean_summary_to_block_token_ratio=("summary_to_block_token_ratio", "mean"),
        trace_median_summary_to_block_token_ratio=("summary_to_block_token_ratio", "median"),
        trace_high_compression_share=("high_token_compression", "mean"),
    )
    .reset_index()
)

# The total trace compression ratio compares all summary tokens against all
# original block tokens within the same trace.
trace_compression["trace_total_summary_to_block_token_ratio"] = (
    trace_compression["trace_summary_tokens"]
    / trace_compression["trace_block_tokens"]
)

trace_compression.head()

In [25]:
# Merge trace-level compression summaries onto the original trace-level table.
df_traces_full = df_traces_full.merge(
    trace_compression,
    on="trace_id",
    how="left",
)

df_traces_full.shape

(228557, 25)

In [26]:
# Define a trace-level high-compression target using the lowest quartile of the
# total trace compression ratio.
trace_compression_threshold = (
    df_traces_full["trace_total_summary_to_block_token_ratio"].quantile(0.25)
)

df_traces_full["trace_high_token_compression"] = (
    df_traces_full["trace_total_summary_to_block_token_ratio"] <= trace_compression_threshold
).astype(int)

trace_compression_threshold, df_traces_full["trace_high_token_compression"].value_counts(normalize=True).round(4)

(np.float64(0.1327609995400889),
 trace_high_token_compression
 0    0.75
 1    0.25
 Name: proportion, dtype: float64)

In [27]:
# Save labeled feature tables for downstream full-data modeling notebooks.
df_blocks_full.to_parquet(
    OUTPUT_DIR / "blocks_features_full_labeled.parquet",
    index=False,
)

df_traces_full.to_parquet(
    OUTPUT_DIR / "traces_features_full_labeled.parquet",
    index=False,
)

In [28]:
pd.read_parquet(OUTPUT_DIR / "blocks_features_full_labeled.parquet").shape

(2013510, 22)

In [29]:
pd.read_parquet(OUTPUT_DIR / "traces_features_full_labeled.parquet").shape

(228557, 26)

## Data Loading and Engineering Outputs

The full OpenMementos feature build produced two local labeled parquet files:

- `blocks_features_full_labeled.parquet`: block-level features and the block-level `high_token_compression` target.
- `traces_features_full_labeled.parquet`: trace-level features and the trace-level `trace_high_token_compression` target.

The block-level table has one row per `(reasoning block, summary)` pair. The trace-level table has one row per original reasoning trace.

These files are saved under the ignored `data/` directory and are not committed to Git.

## Data Alignment Checks

This section checks that rows from the original streamed dataset align with the engineered trace-level and block-level tables.

During feature construction, `trace_id` is assigned according to streaming order. Therefore, the original dataset row at position `i` should correspond to `trace_id == i` in both the trace table and the block table.

We inspect a few fixed trace IDs by comparing original metadata, parsed block counts, engineered trace features, and block-level rows.

In [33]:
from src.reasoning_compression.features import parse_response

In [ ]:
# Use fixed early traces so streaming alignment checks stay cheap and reproducible.
trace_ids_to_check = [0, 1, 2]

In [35]:
def get_original_rows_by_position(
    trace_ids: list[int],
) -> dict[int, Mapping[str, object]]:
    if not trace_ids:
        raise ValueError("trace_ids must contain at least one trace ID.")
    if any(trace_id < 0 for trace_id in trace_ids):
        raise ValueError("trace_ids must be non-negative.")

    # The dataset is streamed in order, so row position i should match trace_id i.
    target_ids = set(trace_ids)
    max_id = max(target_ids)

    ds_stream = load_dataset(
        DATASET_ID,
        split=SPLIT,
        streaming=True,
    )

    original_rows = {}

    for i, row in enumerate(islice(ds_stream, max_id + 1)):
        if i in target_ids:
            original_rows[i] = row

        if len(original_rows) == len(target_ids):
            break

    missing_trace_ids = target_ids.difference(original_rows)
    if missing_trace_ids:
        raise ValueError(
            "Could not retrieve original rows for trace_ids: "
            f"{sorted(missing_trace_ids)}"
        )

    return original_rows

In [ ]:
original_rows = get_original_rows_by_position(trace_ids_to_check)

original_rows.keys()

In [ ]:
alignment_rows = []

for trace_id in trace_ids_to_check:
    original = original_rows[trace_id]
    parsed = parse_response(original["response"])

    trace_row = (
        df_traces_full
        .query("trace_id == @trace_id")
        .iloc[0]
    )

    block_rows = (
        df_blocks_full
        .query("trace_id == @trace_id")
        .sort_values("block_index")
    )

    alignment_rows.append({
        "trace_id": trace_id,
        "original_domain": original["domain"],
        "engineered_domain": trace_row["domain"],
        "original_source": original["source"],
        "engineered_source": trace_row["source"],
        "original_difficulty": original["difficulty"],
        "engineered_difficulty": trace_row["difficulty"],
        "parsed_n_blocks": parsed["n_blocks"],
        "trace_n_blocks": trace_row["n_blocks"],
        "block_table_rows": len(block_rows),
        "parsed_n_summaries": parsed["n_summaries"],
        "trace_n_summaries": trace_row["n_summaries"],
    })

df_alignment_check = pd.DataFrame(alignment_rows)

df_alignment_check

In [ ]:
df_alignment_check.assign(
    domain_match=lambda df: df["original_domain"] == df["engineered_domain"],
    source_match=lambda df: df["original_source"] == df["engineered_source"],
    difficulty_match=lambda df: df["original_difficulty"] == df["engineered_difficulty"],
    block_count_match=lambda df: df["parsed_n_blocks"] == df["trace_n_blocks"],
    block_rows_match=lambda df: df["parsed_n_blocks"] == df["block_table_rows"],
    summary_count_match=lambda df: df["parsed_n_summaries"] == df["trace_n_summaries"],
)

In [ ]:
TRACE_ID_INSPECT = trace_ids_to_check[0]

df_traces_full.query("trace_id == @TRACE_ID_INSPECT")

In [ ]:
df_blocks_full.query("trace_id == @TRACE_ID_INSPECT").sort_values("block_index")[[
    "trace_id",
    "block_index",
    "block_tokens",
    "summary_tokens",
    "summary_to_block_token_ratio",
    "token_compression_savings",
    "high_token_compression",
]]

In [ ]:
original = original_rows[TRACE_ID_INSPECT]
parsed = parse_response(original["response"])

print("Problem preview:")
print(original["problem"][:1000])

print("\nFirst block preview:")
print(parsed["blocks"][0][:1000])

print("\nFirst summary preview:")
print(parsed["summaries"][0][:1000])